# Agente de Análise de Arquivos Potencialmente Maliciosos
### Componentes:
Clara Virgínia Nascimento Santos - 20240002212 \
Heloísa Maria Alves Severo - 20240057945

### 0. Pipeline Inicial do Projeto
![https://miro.com/app/board/uXjVHjbORDo=/?share_link_id=649144196376](pipeline.png)

Link: https://miro.com/app/board/uXjVHjbORDo=/?share_link_id=649144196376

### 1. Import do csv com 10 casos e conversao para JSON

In [248]:
from agentkit import Agent, LLM, LLMAPI, tool
import pandas as pd
import json
from typing import Callable, get_type_hints
import torch

In [249]:
# API do Groq vou usar como variável de ambiente
import os
api_key = os.getenv("GROQ_API_KEY")

print(api_key is not None)  # Verifica se a variável de ambiente foi carregada corretamente


True


In [250]:
llm = LLMAPI(
    model="openai/gpt-oss-20b", #20 bilhoes de parametros
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1",
    temperature=0.0,
    max_tokens=1024
    )

In [251]:
resposta = llm.invoke("Responda com a palavra Clara.")
print(resposta)

Clara


In [252]:
# leitura do casos.csv
df = pd.read_csv('casos.csv')
df.head(10)

,id,nome_arquivo,extensao,tamanho_kb,origem,hash,assinatura_digital,criou_processo,alterou_registro,criou_arquivo,conexao_externa,resultado_esperado
0,1,relatorio.pdf,.pdf,850,download,hash_benigno_001,valida,False,False,False,False,BENIGNO
1,2,invoice.exe,.exe,2450,email,hash_malicioso_001,invalida,True,True,False,True,MALICIOSO
2,3,foto.jpg,.jpg,3200,pendrive,hash_desconhecido_001,ausente,False,False,False,False,BENIGNO
3,4,update.exe,.exe,1800,site_desconhecido,hash_desconhecido_002,ausente,True,True,True,True,MALICIOSO
4,5,documento.docx,.docx,540,email,hash_benigno_002,ausente,False,False,False,False,BENIGNO
5,6,setup.exe,.exe,5200,site_oficial,hash_desconhecido_003,valida,True,False,False,True,SUSPEITO
6,7,script.ps1,.ps1,45,email,hash_desconhecido_004,ausente,True,True,False,True,MALICIOSO
7,8,imagem.png,.png,1200,download,hash_benigno_003,ausente,False,False,False,False,BENIGNO
8,9,game_crack.exe,.exe,7600,site_desconhecido,hash_desconhecido_005,ausente,True,False,True,True,MALICIOSO
9,10,backup.zip,.zip,15000,rede_interna,hash_desconhecido_006,ausente,False,False,False,False,SUSPEITO


In [253]:
# converter para dict e separar o resultado_separado
casos = []

for _, linha in df.iterrows(): # percorre o df linha por linha
    arquivo = linha.drop("resultado_esperado").to_dict() #forma o arquivo sem o resultado_esperado

    caso = { # é o caso completo
        "arquivo": arquivo,
        "resultado_esperado": linha["resultado_esperado"]
    }
    casos.append(caso)

len(casos)
casos[0]

{'arquivo': {'id': 1,
  'nome_arquivo': 'relatorio.pdf',
  'extensao': '.pdf',
  'tamanho_kb': 850,
  'origem': 'download',
  'hash': 'hash_benigno_001',
  'assinatura_digital': 'valida',
  'criou_processo': False,
  'alterou_registro': False,
  'criou_arquivo': False,
  'conexao_externa': False},
 'resultado_esperado': 'BENIGNO'}

In [254]:
# converter para json o arquivo a ser enviado pro agente
arquivo = casos[0]["arquivo"] # convertendo o primeiro caso sem o resultado_esperado para json
arquivo_json = json.dumps(
    arquivo,
    ensure_ascii=False,
    indent=4
    )

print(arquivo_json)

{
    "id": 1,
    "nome_arquivo": "relatorio.pdf",
    "extensao": ".pdf",
    "tamanho_kb": 850,
    "origem": "download",
    "hash": "hash_benigno_001",
    "assinatura_digital": "valida",
    "criou_processo": false,
    "alterou_registro": false,
    "criou_arquivo": false,
    "conexao_externa": false
}


### 2. Import do indicadores.csv

In [255]:
df_indicadores = pd.read_csv('indicadores.csv')

df_indicadores.head()

,hash,status,descricao
0,hash_benigno_001,benigno,Arquivo conhecido como legítimo e sem indícios...
1,hash_malicioso_001,malicioso,Arquivo previamente identificado como malicios...
2,hash_benigno_002,benigno,Arquivo conhecido como legítimo e sem indícios...
3,hash_benigno_003,benigno,Arquivo conhecido como legítimo e sem indícios...


In [256]:
# testar consulta manual
hash_consultado = "hash_malicioso_001"

resultado = df_indicadores[df_indicadores["hash"] == hash_consultado]
resultado

,hash,status,descricao
1,hash_malicioso_001,malicioso,Arquivo previamente identificado como malicios...


### 3. Construcao das ferramentas

In [257]:
# tool1 -> consultar indicadores conhecidos
@tool
def consultar_indicador(id_arquivo: int) -> str: # vai receber o id_arquivo pq o hash e outras informações não serão enviadas ao agente inicialmente
    """Consulta se o hash de um arquivo possui indicador conhecido."""

    # 1. Procura o arquivo no dataframe de casos
    resultado_arquivo = df[df["id"] == id_arquivo]

    # 2. Verifica se o id realmente existe
    if len(resultado_arquivo) == 0:
        return f"Arquivo com ID {id_arquivo} não encontrado."

    # 3. Obtem o hash desse arquivo
    hash_arquivo = resultado_arquivo.iloc[0]["hash"]

    # 4. Procura o hash na base de indicadores
    resultado_indicador = df_indicadores[
        df_indicadores["hash"] == hash_arquivo
    ]

    # 5. Retorna a evidência encontrada
    if len(resultado_indicador) > 0:
        indicador = resultado_indicador.to_dict(orient="records")[0]
        return str(indicador)

    return "Nenhum indicador conhecido para o hash deste arquivo."

In [258]:
# tool2 -> consultar os metadados (nome, extensao, tamanho, origem, assinatura)
@tool
def analisar_metadados(id_arquivo: int) -> str:
    """Analisa os metadados estaticos de um arquivo e retorna evidencias relevantes."""

    # Procura o arquivo no dataframe
    resultado_arquivo = df[df["id"] == id_arquivo]

    if len(resultado_arquivo) == 0:
        return f"Arquivo com ID {id_arquivo} nao encontrado."

    # Pega a linha correspondente ao arquivo
    arquivo = resultado_arquivo.iloc[0]

    # Obtem os metadados necessários
    nome_arquivo = arquivo["nome_arquivo"]
    extensao = arquivo["extensao"]
    tamanho_kb = arquivo["tamanho_kb"]
    origem = arquivo["origem"]
    assinatura_digital = arquivo["assinatura_digital"]

    evidencias = []

    if extensao.lower() in [".exe", ".ps1"]:
        evidencias.append(
            f"Extensao executavel ou de script: {extensao}."
        )

    if origem.lower() in ["email", "download", "internet"]:
        evidencias.append(
            f"Arquivo proveniente de origem potencialmente não confiavel: {origem}."
        )

    if assinatura_digital.lower() in ["invalida", "ausente"]:
        evidencias.append(
            f"Assinatura digital {assinatura_digital}."
        )

    termos_suspeitos = ["crack", "keygen", "invoice"]

    for termo in termos_suspeitos:
        if termo in nome_arquivo.lower():
            evidencias.append(
                f"Nome do arquivo contém termo potencialmente suspeito: {termo}."
            )
            break

    if not evidencias:
        return "Nenhuma evidencia suspeita identificada nos metadados."

    return " ".join(evidencias)


In [259]:
# tool3 -> acoes observadas durante a análises
@tool
def analisar_comportamento(id_arquivo: int) -> str:
    """Analisa os comportamentos observados de um arquivo e retorna evidencias relevantes."""

    # Procura o arquivo no dataframe
    resultado_arquivo = df[df["id"] == id_arquivo]

    if len(resultado_arquivo) == 0:
        return f"Arquivo com ID {id_arquivo} nao encontrado."

    # Pega a linha correspondente ao arquivo
    arquivo = resultado_arquivo.iloc[0]

    # Obtem os comportamentos observados
    criou_processo = arquivo["criou_processo"]
    alterou_registro = arquivo["alterou_registro"]
    criou_arquivo = arquivo["criou_arquivo"]
    conexao_externa = arquivo["conexao_externa"]

    evidencias = []

    if criou_processo:
        evidencias.append(
            "O arquivo criou um novo processo."
        )

    if alterou_registro:
        evidencias.append(
            "O arquivo realizou alteração no registro do sistema."
        )

    if criou_arquivo:
        evidencias.append(
            "O arquivo criou outro arquivo no sistema."
        )

    if conexao_externa:
        evidencias.append(
            "O arquivo realizou conexao externa."
        )

    if not evidencias:
        return "Nenhum comportamento suspeito foi observado."

    return " ".join(evidencias)

### 4. Agente


In [260]:
agente = Agent(
    llm=llm,
    tools=[consultar_indicador, analisar_metadados, analisar_comportamento],
    max_steps=5
)


# Criacao do system prompt -> usado o chatgpt para criar o prompt
SYSTEM_PROMPT = """
Você é um agente de triagem de arquivos potencialmente maliciosos.

Sua tarefa é analisar um único arquivo por execução e classificá-lo como:
- BENIGNO
- SUSPEITO
- MALICIOSO

Você receberá apenas a identificação básica do arquivo.
Para obter informações técnicas e evidências, utilize as ferramentas disponíveis.

As ferramentas permitem investigar:
- indicadores conhecidos associados ao arquivo;
- metadados estáticos;
- comportamentos observados.

Decida dinamicamente quais ferramentas utilizar, em qual ordem e quando possui
evidências suficientes para encerrar a investigação.

A ausência de evidência não deve ser confundida com evidência de ausência.

Se uma dimensão de análise não foi investigada por uma ferramenta, não conclua
que não existem sinais suspeitos nessa dimensão.

Quando as evidências obtidas forem insuficientes ou ambíguas para sustentar
com segurança uma classificação, utilize ferramentas adicionais antes de
encerrar a investigação.

Não assuma que um único sinal isolado determina necessariamente a classificação.
Considere o conjunto das evidências obtidas durante a investigação.

Fundamente sua decisão exclusivamente nas informações fornecidas na entrada
e nas evidências retornadas pelas ferramentas.
Não invente informações que não tenham sido observadas.
Não trate a base de indicadores como uma base pública ou externa.
Não afirme que uma análise foi realizada se ela não foi executada pelas
ferramentas disponíveis.

Ao concluir, informe:
1. a classificação;
2. as principais evidências que sustentam a decisão.
"""

#### 4.1. Configurar structured contract e Reflection


In [261]:
# Imports
from pydantic import BaseModel
from typing import Literal

class ResultadoTriagem(BaseModel): 
    classificacao: Literal["BENIGNO", "SUSPEITO", "MALICIOSO"]
    evidencias: list[str]

# Estruturação do reflextion
class ResultadoReflexao(BaseModel):
    aprovado: bool
    problemas: list[str]
    sugestao_correcao: str

In [262]:
def extrair_resposta_final(historico: list) -> str:
    """Extrai o conteudo da última mensagem do assistente."""

    for mensagem in reversed(historico):
        if mensagem.get("role") == "assistant" and mensagem.get("content"):
            return mensagem["content"]

    return ""

def extrair_evidencias_tools(historico: list) -> list[str]:
    """Extrai as evidências realmente retornadas pelas ferramentas."""

    evidencias = []

    for mensagem in historico:
        if mensagem.get("role") == "tool":
            evidencias.append(
                f'{mensagem.get("name")}: {mensagem.get("content")}'
            )

    return evidencias


### 4.2. Pipeline de Triagem

In [263]:
def executar_triagem(arquivo: dict) -> dict:
    """Executa a triagem completa de um unico arquivo."""

    entrada = f"""
ID: {arquivo["id"]}
Nome do arquivo: {arquivo["nome_arquivo"]}
"""

    prompt = f"""
{SYSTEM_PROMPT}

ARQUIVO A SER ANALISADO:

{entrada}
"""

    historico = agente.run(prompt)

    texto_final = extrair_resposta_final(historico)

    resultado_estruturado = llm.generate_structured(
        f"""
    Converta o resultado da triagem abaixo para o formato estruturado solicitado.

    Não adicione novas evidencias.
    Utilize somente as informações presentes no resultado da triagem.

    RESULTADO DA TRIAGEM:
    {texto_final}
    """,
        ResultadoTriagem
    )
    evidencias_tools = extrair_evidencias_tools(historico)

    prompt_reflexao = f"""
    Você é o revisor de um agente de triagem de arquivos em um ambiente simulado.

    As únicas classificações permitidas são:
    - BENIGNO
    - SUSPEITO
    - MALICIOSO

    Sua tarefa NÃO é realizar uma nova investigação.
    Revise a conclusão produzida pelo agente usando exclusivamente
    as evidências retornadas pelas ferramentas.

    Considere três situações:

    1. EVIDÊNCIA SUPORTADA:
    A afirmação aparece explicitamente nas evidências das ferramentas.

    2. PARÁFRASE VÁLIDA:
    A afirmação apenas reescreve uma evidência da ferramenta sem acrescentar
    novo significado. Isso NÃO deve ser considerado um problema.

    3. INFERÊNCIA NÃO SUPORTADA:
    A afirmação acrescenta detalhes, causas, intenções, destinos, consequências
    ou interpretações que não aparecem nas evidências das ferramentas.
    Isso DEVE ser considerado um problema.

    Avalie:
    - se a classificação é coerente com as evidências reais das ferramentas;
    - se existem contradições entre evidências e classificação;
    - se as evidências declaradas são suportadas pelas ferramentas;
    - se existem inferências não suportadas.

    Não solicite novas análises.
    Não sugira novas ferramentas.
    Não utilize informações externas ao ambiente simulado.
    Não proponha novas categorias.
    Não invente novas evidências.

    Se encontrar uma inferência não suportada, marque aprovado como False,
    descreva exatamente o trecho problemático e sugira removê-lo ou reescrevê-lo
    usando somente a evidência disponível.

    EVIDÊNCIAS REAIS RETORNADAS PELAS FERRAMENTAS:
    {evidencias_tools}

    RESULTADO PRODUZIDO PELO AGENTE:

    Classificação:
    {resultado_estruturado.classificacao}

    Evidências declaradas:
    {resultado_estruturado.evidencias}
    """

    resultado_reflexao = llm.generate_structured(
        prompt_reflexao,
        ResultadoReflexao
    )

    return {
        "historico": historico,
        "texto_final": texto_final,
        "resultado_estruturado": resultado_estruturado,
        "evidencias_tools": evidencias_tools,
        "resultado_reflexao": resultado_reflexao
    }

### 5. Testes Antes dos Turnos

#### 5.1. Teste 1


In [264]:
# Gabarito para o primeiro caso
caso = casos[0]
arquivo = caso["arquivo"]


entrada = f"""
Analise o seguinte arquivo:
ID: {arquivo['id']}
Nome do arquivo: {arquivo['nome_arquivo']}"""

print(entrada)


Analise o seguinte arquivo:
ID: 1
Nome do arquivo: relatorio.pdf


In [265]:
prompt = f"""
{SYSTEM_PROMPT}
ARQUIVO A SER ANALISADO: {entrada}
"""

resposta = agente.run(prompt)
print(resposta)

[{'role': 'user', 'content': '\n\nVocê é um agente de triagem de arquivos potencialmente maliciosos.\n\nSua tarefa é analisar um único arquivo por execução e classificá-lo como:\n- BENIGNO\n- SUSPEITO\n- MALICIOSO\n\nVocê receberá apenas a identificação básica do arquivo.\nPara obter informações técnicas e evidências, utilize as ferramentas disponíveis.\n\nAs ferramentas permitem investigar:\n- indicadores conhecidos associados ao arquivo;\n- metadados estáticos;\n- comportamentos observados.\n\nDecida dinamicamente quais ferramentas utilizar, em qual ordem e quando possui\nevidências suficientes para encerrar a investigação.\n\nA ausência de evidência não deve ser confundida com evidência de ausência.\n\nSe uma dimensão de análise não foi investigada por uma ferramenta, não conclua\nque não existem sinais suspeitos nessa dimensão.\n\nQuando as evidências obtidas forem insuficientes ou ambíguas para sustentar\ncom segurança uma classificação, utilize ferramentas adicionais antes de\nen

#### 5.2. Teste 2

In [266]:
caso = casos[5]

arquivo = caso["arquivo"]

entrada = f"""
Analise o seguinte arquivo:

ID: {arquivo["id"]}
Nome do arquivo: {arquivo["nome_arquivo"]}
"""

prompt = f"""
{SYSTEM_PROMPT}

ARQUIVO A SER ANALISADO:

{entrada}
"""

resposta = agente.run(prompt)

print(resposta)


[{'role': 'user', 'content': '\n\nVocê é um agente de triagem de arquivos potencialmente maliciosos.\n\nSua tarefa é analisar um único arquivo por execução e classificá-lo como:\n- BENIGNO\n- SUSPEITO\n- MALICIOSO\n\nVocê receberá apenas a identificação básica do arquivo.\nPara obter informações técnicas e evidências, utilize as ferramentas disponíveis.\n\nAs ferramentas permitem investigar:\n- indicadores conhecidos associados ao arquivo;\n- metadados estáticos;\n- comportamentos observados.\n\nDecida dinamicamente quais ferramentas utilizar, em qual ordem e quando possui\nevidências suficientes para encerrar a investigação.\n\nA ausência de evidência não deve ser confundida com evidência de ausência.\n\nSe uma dimensão de análise não foi investigada por uma ferramenta, não conclua\nque não existem sinais suspeitos nessa dimensão.\n\nQuando as evidências obtidas forem insuficientes ou ambíguas para sustentar\ncom segurança uma classificação, utilize ferramentas adicionais antes de\nen

In [267]:
texto_final = extrair_resposta_final(resposta)

resultado_estruturado = llm.generate_structured(
    f"""
Converta o resultado da triagem abaixo para o formato estruturado solicitado.

Não adicione novas evidencias.
Utilize somente as informacoes presentes no resultado da triagem.

RESULTADO DA TRIAGEM:
{texto_final}
""",
    ResultadoTriagem
)

print(resultado_estruturado)


classificacao='SUSPEITO' evidencias=['Indicadores conhecidos: Nenhum indicador de malware foi encontrado no hash do arquivo (consulta `consultar_indicador`).', 'Metadados estáticos: O arquivo possui extensão `.exe`, indicando que é um executável, mas não há outras informações de metadados que apontem para benignidade ou malignidade.', 'Comportamento observado: O arquivo criou um novo processo (`analisar_comportamento`). O arquivo estabeleceu uma conexão externa (`analisar_comportamento`).']


In [268]:
print("Classificacao:", resultado_estruturado.classificacao)
print("Evidencias:", resultado_estruturado.evidencias)
print("Tipo:", type(resultado_estruturado))

Classificacao: SUSPEITO
Evidencias: ['Indicadores conhecidos: Nenhum indicador de malware foi encontrado no hash do arquivo (consulta `consultar_indicador`).', 'Metadados estáticos: O arquivo possui extensão `.exe`, indicando que é um executável, mas não há outras informações de metadados que apontem para benignidade ou malignidade.', 'Comportamento observado: O arquivo criou um novo processo (`analisar_comportamento`). O arquivo estabeleceu uma conexão externa (`analisar_comportamento`).']
Tipo: <class '__main__.ResultadoTriagem'>


In [269]:
evidencias_tools = extrair_evidencias_tools(resposta)

for evidencia in evidencias_tools:
    print(evidencia)

consultar_indicador: Nenhum indicador conhecido para o hash deste arquivo.
analisar_metadados: Extensao executavel ou de script: .exe.
analisar_comportamento: O arquivo criou um novo processo. O arquivo realizou conexao externa.


In [270]:
# texto produzido com chatgpt
prompt_reflexao = f"""
Você é o revisor de um agente de triagem de arquivos em um ambiente simulado.

As únicas classificações permitidas são:
- BENIGNO
- SUSPEITO
- MALICIOSO

Sua tarefa NÃO é realizar uma nova investigação.
Sua tarefa é revisar a conclusão produzida pelo agente com base exclusivamente
nas evidências retornadas pelas ferramentas.

Avalie:
- se a classificação é coerente com as evidências reais das ferramentas;
- se existem contradições entre evidências e classificação;
- se as evidências declaradas no resultado são sustentadas pelas ferramentas;
- se foram introduzidas informações não observadas pelas ferramentas.

Não solicite novas análises.
Não sugira novas ferramentas.
Não utilize critérios ou informações externas ao ambiente simulado.
Não proponha novas categorias de classificação.
Não invente novas evidências.

EVIDÊNCIAS REAIS RETORNADAS PELAS FERRAMENTAS:
{evidencias_tools}

RESULTADO PRODUZIDO PELO AGENTE:

Classificação:
{resultado_estruturado.classificacao}

Evidências declaradas:
{resultado_estruturado.evidencias}
"""

resultado_reflexao = llm.generate_structured(
    prompt_reflexao,
    ResultadoReflexao
)



#### 5.3. Teste 3


In [276]:
caso = casos[5]

resultado = executar_triagem(caso["arquivo"])

In [277]:
print("Arquivo:", caso["arquivo"]["nome_arquivo"])
print("Esperado:", caso["resultado_esperado"])

print("\n### RESULTADO DA TRIAGEM ###")
print("Obtido:", resultado["resultado_estruturado"].classificacao)
print("Evidencias:", resultado["resultado_estruturado"].evidencias)

print("\n### TOOLS ###")
for evidencia in resultado["evidencias_tools"]:
    print(evidencia)

print("\n### REFLECTION ###")
print("Aprovado:", resultado["resultado_reflexao"].aprovado)
print("Problemas:", resultado["resultado_reflexao"].problemas)
print(
    "Sugestao:",
    resultado["resultado_reflexao"].sugestao_correcao
)

Arquivo: setup.exe
Esperado: SUSPEITO

### RESULTADO DA TRIAGEM ###
Obtido: SUSPEITO
Evidencias: ['O arquivo `setup.exe` criou um novo processo.', 'O arquivo estabeleceu uma conexão externa.', 'Nenhum indicador conhecido foi associado ao hash do arquivo.', 'O arquivo possui extensão `.exe`, típica de instaladores, mas isso não garante benignidade.']

### TOOLS ###
consultar_indicador: Nenhum indicador conhecido para o hash deste arquivo.
analisar_metadados: Extensao executavel ou de script: .exe.
analisar_comportamento: O arquivo criou um novo processo. O arquivo realizou conexao externa.

### REFLECTION ###
Aprovado: False
Problemas: ["O trecho 'O arquivo possui extensão `.exe`, típica de instaladores, mas isso não garante benignidade.' contém inferência não suportada."]
Sugestao: Remover a parte 'típica de instaladores, mas isso não garante benignidade' ou reescrever apenas como 'O arquivo possui extensão `.exe`', que é a evidência fornecida.


### 6. Avaliação nos 10 casos

In [279]:
import time #por causa do limite do groq

In [281]:
resultados_testes = []

for caso in casos:

    arquivo = caso["arquivo"]

    resultado = executar_triagem(arquivo)

    esperado = caso["resultado_esperado"]
    obtido = resultado["resultado_estruturado"].classificacao

    passou = esperado == obtido

    resultados_testes.append({
        "id": arquivo["id"],
        "arquivo": arquivo["nome_arquivo"],
        "esperado": esperado,
        "obtido": obtido,
        "passou": passou,
        "reflexao_aprovada": resultado["resultado_reflexao"].aprovado
    })

    print(
        f'{arquivo["nome_arquivo"]}: '
        f'esperado={esperado} | obtido={obtido} | passou={passou}'
    )

    time.sleep(100)

relatorio.pdf: esperado=BENIGNO | obtido=BENIGNO | passou=True
invoice.exe: esperado=MALICIOSO | obtido=MALICIOSO | passou=True
foto.jpg: esperado=BENIGNO | obtido=BENIGNO | passou=True
update.exe: esperado=MALICIOSO | obtido=SUSPEITO | passou=False
documento.docx: esperado=BENIGNO | obtido=BENIGNO | passou=True
setup.exe: esperado=SUSPEITO | obtido=SUSPEITO | passou=True
script.ps1: esperado=MALICIOSO | obtido=MALICIOSO | passou=True
imagem.png: esperado=BENIGNO | obtido=BENIGNO | passou=True
game_crack.exe: esperado=MALICIOSO | obtido=MALICIOSO | passou=True
backup.zip: esperado=SUSPEITO | obtido=BENIGNO | passou=False


#### 6.1. Medir a acurácia

In [282]:
acertos = sum(resultado["passou"] for resultado in resultados_testes)
total = len(resultados_testes)
acuracia = acertos / total

print(f"Acertos: {acertos}/{total}")
print(f"Acuracia: {acuracia:.2%}")

Acertos: 8/10
Acuracia: 80.00%


#### 6.2. Analisando resultado do caso backup.zip

In [284]:
caso_backup = casos[9]

resultado_backup = executar_triagem(caso_backup["arquivo"])

print("Obtido:", resultado_backup["resultado_estruturado"].classificacao)

print("\n### TOOLS ###")
for evidencia in resultado_backup["evidencias_tools"]:
    print(evidencia)

print("\n### REFLECTION ###")
print(resultado_backup["resultado_reflexao"])

Obtido: BENIGNO

### TOOLS ###
consultar_indicador: Nenhum indicador conhecido para o hash deste arquivo.
analisar_metadados: Assinatura digital ausente.
analisar_comportamento: Nenhum comportamento suspeito foi observado.

### REFLECTION ###
aprovado=True problemas=[] sugestao_correcao=''


#### 6.3. Analisando resultado do caso update.exe

In [285]:
caso_update = casos[3]

resultado_update = executar_triagem(caso_update["arquivo"])

print("Esperado:", caso_update["resultado_esperado"])
print("Obtido:", resultado_update["resultado_estruturado"].classificacao)

print("\n### TOOLS ###")
for evidencia in resultado_update["evidencias_tools"]:
    print(evidencia)

print("\n### REFLECTION ###")
print(resultado_update["resultado_reflexao"])

Esperado: MALICIOSO
Obtido: SUSPEITO

### TOOLS ###
consultar_indicador: Nenhum indicador conhecido para o hash deste arquivo.
analisar_metadados: Extensao executavel ou de script: .exe. Assinatura digital ausente.
analisar_comportamento: O arquivo criou um novo processo. O arquivo realizou alteração no registro do sistema. O arquivo criou outro arquivo no sistema. O arquivo realizou conexao externa.

### REFLECTION ###
aprovado=True problemas=[] sugestao_correcao=''


### 7. Análise

O agente obteve 8 acertos em 10 casos, resultando em uma acurácia de 80%. Os dois erros ocorreram nos arquivos `update.exe` e `backup.zip`.

No caso do `update.exe`, o resultado esperado era "MALICIOSO", mas o agente classificou o arquivo como "SUSPEITO". A investigação utilizou as três ferramentas disponíveis. Não foi encontrado um indicador conhecido para o hash, enquanto a análise de metadados identificou um arquivo executável sem assinatura digital. A análise comportamental identificou criação de processo, alteração do registro, criação de outro arquivo e conexão externa. Portanto, o erro não ocorreu por falta de investigação, mas pela interpretação das evidências: o agente atribuiu peso insuficiente à combinação de múltiplos comportamentos suspeitos e produziu uma classificação menos severa que a esperada.

No caso do `backup.zip`, o resultado esperado era "SUSPEITO", mas o agente classificou o arquivo como "BENIGNO". Também foram utilizadas as três ferramentas. Não havia indicador conhecido associado ao hash e nenhum comportamento suspeito foi observado, porém a assinatura digital estava ausente. Nesse caso, o agente atribuiu pouco peso à ausência da assinatura diante das demais evidências e acabou classificando o arquivo como benigno.

Nos dois erros, a etapa de "Reflection" aprovou os resultados. Isso ocorre porque a "Reflection" foi projetada para verificar se a conclusão está fundamentada nas evidências efetivamente retornadas pelas ferramentas e se existem afirmações não suportadas, e não para comparar a classificação com o gabarito. Assim, uma resposta pode estar fundamentada nas evidências disponíveis e ainda produzir uma classificação diferente da esperada.

Os resultados mostram que o agente consegue realizar investigações em múltiplas etapas e utilizar diferentes ferramentas, mas a interpretação do peso e da combinação das evidências ainda pode produzir classificações incorretas.